In [ ]:
pip install transformers datasets seqeval torch accelerate


In [12]:
# ============================================================
#   BERT NER with WikiANN Dataset — Complete Code
# ============================================================

import numpy as np
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import f1_score, classification_report

In [13]:
# ─────────────────────────────────────────────
# 1. LOAD DATASET
# ─────────────────────────────────────────────

dataset = load_dataset("wikiann", "en")

print("Dataset loaded:")
print(dataset)
print("\nSample training example:")
print(dataset["train"][0])

Dataset loaded:
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})

Sample training example:
{'tokens': ['R.H.', 'Saunders', '(', 'St.', 'Lawrence', 'River', ')', '(', '968', 'MW', ')'], 'ner_tags': [3, 4, 0, 3, 4, 4, 0, 0, 0, 0, 0], 'langs': ['en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en', 'en'], 'spans': ['ORG: R.H. Saunders', 'ORG: St. Lawrence River']}


In [14]:
# ─────────────────────────────────────────────
# 2. LABEL SETUP
# ─────────────────────────────────────────────

label_names = dataset["train"].features["ner_tags"].feature.names
id2label    = {i: label for i, label in enumerate(label_names)}
label2id    = {label: i for i, label in enumerate(label_names)}

print(f"\nLabel names ({len(label_names)}): {label_names}")
# ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


Label names (7): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [15]:
# ─────────────────────────────────────────────
# 3. TOKENIZER
# ─────────────────────────────────────────────

MODEL_CHECKPOINT = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [16]:
# ─────────────────────────────────────────────
# 4. TOKENIZE + ALIGN LABELS
# ─────────────────────────────────────────────

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids          = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids         = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["tokens", "ner_tags", "langs", "spans"]
)

print("\nTokenization complete.")
print(tokenized_dataset)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]


Tokenization complete.
DatasetDict({
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [17]:
# ─────────────────────────────────────────────
# 5. MODEL
# ─────────────────────────────────────────────

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print(f"\nModel loaded: {MODEL_CHECKPOINT}")
print(f"Number of labels: {len(label_names)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca


Model loaded: bert-base-cased
Number of labels: 7


In [18]:
# ─────────────────────────────────────────────
# 6. EVALUATION METRIC
# ─────────────────────────────────────────────

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions    = np.argmax(logits, axis=-1)

    true_labels, true_predictions = [], []
    for prediction, label in zip(predictions, labels):
        true_label_row, true_pred_row = [], []
        for pred_id, label_id in zip(prediction, label):
            if label_id == -100:
                continue
            true_label_row.append(label_names[label_id])
            true_pred_row.append(label_names[pred_id])
        true_labels.append(true_label_row)
        true_predictions.append(true_pred_row)

    return {
        "f1": f1_score(true_labels, true_predictions),
    }

In [24]:
# ─────────────────────────────────────────────
# 7. TRAINING
# ─────────────────────────────────────────────

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir                  = "bert-ner-wikiann",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    learning_rate               = 2e-5,
    num_train_epochs            = 3,
    weight_decay                = 0.01,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    logging_steps               = 100,
    push_to_hub                 = False,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset  = tokenized_dataset["validation"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

print("\nStarting training...")
trainer.train()


Starting training...


Epoch,Training Loss,Validation Loss,F1
1,0.123044,0.280589,0.824852
2,0.102858,0.313483,0.827797
3,0.060472,0.348485,0.838946


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3750, training_loss=0.0838286698659261, metrics={'train_runtime': 570.5875, 'train_samples_per_second': 105.155, 'train_steps_per_second': 6.572, 'total_flos': 959590390953216.0, 'train_loss': 0.0838286698659261, 'epoch': 3.0})

In [25]:
# ─────────────────────────────────────────────
# 8. EVALUATE ON TEST SET
# ─────────────────────────────────────────────

print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_dataset["test"])
print(f"Test F1 Score: {test_results['eval_f1']:.4f}")

# Detailed per-entity report
def full_report(trainer, dataset):
    output      = trainer.predict(dataset)
    logits      = output.predictions
    labels      = output.label_ids
    predictions = np.argmax(logits, axis=-1)

    true_labels, true_predictions = [], []
    for prediction, label in zip(predictions, labels):
        true_label_row, true_pred_row = [], []
        for pred_id, label_id in zip(prediction, label):
            if label_id == -100:
                continue
            true_label_row.append(label_names[label_id])
            true_pred_row.append(label_names[pred_id])
        true_labels.append(true_label_row)
        true_predictions.append(true_pred_row)

    print("\nDetailed Classification Report:")
    print(classification_report(true_labels, true_predictions))

full_report(trainer, tokenized_dataset["test"])



Evaluating on test set...


Test F1 Score: 0.8316

Detailed Classification Report:
              precision    recall  f1-score   support

         LOC       0.84      0.86      0.85      4657
         ORG       0.74      0.77      0.75      4745
         PER       0.88      0.91      0.89      4556

   micro avg       0.82      0.84      0.83     13958
   macro avg       0.82      0.85      0.83     13958
weighted avg       0.82      0.84      0.83     13958



In [26]:
# ─────────────────────────────────────────────
# 9. SAVE MODEL
# ─────────────────────────────────────────────

SAVE_DIR = "bert-ner-wikiann"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\nModel saved to ./{SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./bert-ner-wikiann


In [27]:
# ─────────────────────────────────────────────
# 10. LOAD SAVED MODEL FOR INFERENCE
# ─────────────────────────────────────────────

infer_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
infer_model     = AutoModelForTokenClassification.from_pretrained(SAVE_DIR)
infer_model.eval()

print("\nModel loaded for inference.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Model loaded for inference.


In [28]:
# ─────────────────────────────────────────────
# 11. INFERENCE — MANUAL SUBWORD MERGING
# ─────────────────────────────────────────────

def predict_ner(text):
    encoding = infer_tokenizer(
        text,
        return_tensors        = "pt",
        return_offsets_mapping= True,
        truncation            = True,
        max_length            = 512
    )

    # Remove offset_mapping before passing to model (not a model input)
    offset_mapping = encoding.pop("offset_mapping")[0]

    with torch.no_grad():
        outputs = infer_model(**encoding)

    logits       = outputs.logits[0]
    probabilities= torch.softmax(logits, dim=-1)
    predicted_ids= torch.argmax(probabilities, dim=-1)
    tokens       = infer_tokenizer.convert_ids_to_tokens(
                       encoding["input_ids"][0]
                   )

    results        = []
    current_word   = ""
    current_label  = None
    current_scores = []
    current_start  = None

    for token, pred_id, offset, probs in zip(
        tokens, predicted_ids, offset_mapping, probabilities
    ):
        start_char, end_char = offset[0].item(), offset[1].item()

        # Skip special tokens [CLS] and [SEP]
        if start_char == 0 and end_char == 0:
            continue

        label = id2label[pred_id.item()]
        score = probs[pred_id].item()

        if token.startswith("##"):
            # Continuation subword — append characters to current word
            current_word   += token[2:]
            current_scores.append(score)
        else:
            # New word starts — flush the previous word if it was an entity
            if current_word and current_label and current_label != "O":
                results.append({
                    "word"  : current_word,
                    "entity": current_label,
                    "score" : round(
                        sum(current_scores) / len(current_scores), 3
                    ),
                    "start" : current_start,
                })
            # Start tracking the new word using the original text slice
            current_word   = text[start_char:end_char]
            current_label  = label
            current_scores = [score]
            current_start  = start_char

    # Flush the very last word
    if current_word and current_label and current_label != "O":
        results.append({
            "word"  : current_word,
            "entity": current_label,
            "score" : round(sum(current_scores) / len(current_scores), 3),
            "start" : current_start,
        })

    return results


In [29]:
# ─────────────────────────────────────────────
# 12. MERGE BIO TOKENS INTO FULL ENTITY SPANS
# ─────────────────────────────────────────────

def merge_entities(token_predictions):

    merged         = []
    current_entity = None

    for pred in token_predictions:
        bio_tag = pred["entity"]

        if "-" not in bio_tag:
            if current_entity:
                merged.append(current_entity)
            current_entity = None
            continue

        prefix, entity_type = bio_tag.split("-", 1)

        if prefix == "B":
            # New entity begins — flush previous
            if current_entity:
                merged.append(current_entity)
            current_entity = {
                "entity_group": entity_type,
                "word"        : pred["word"],
                "score"       : pred["score"],
                "start"       : pred["start"],
            }

        elif (
            prefix == "I"
            and current_entity
            and current_entity["entity_group"] == entity_type
        ):
            # Continuation of the same entity
            current_entity["word"]  += " " + pred["word"]
            current_entity["score"]  = round(
                (current_entity["score"] + pred["score"]) / 2, 3
            )

        else:
            # Unexpected I- tag with no matching B- — treat as new entity
            if current_entity:
                merged.append(current_entity)
            current_entity = {
                "entity_group": entity_type,
                "word"        : pred["word"],
                "score"       : pred["score"],
                "start"       : pred["start"],
            }

    if current_entity:
        merged.append(current_entity)

    return merged


In [49]:
# ─────────────────────────────────────────────
# 13. DISPLAY FUNCTION WITH THRESHOLD FILTER
# ─────────────────────────────────────────────

def run_ner(text, threshold=0.85):
    print(f"\nInput : {text}")
    print("-" * 62)

    token_preds = predict_ner(text)
    entities    = merge_entities(token_preds)

    # Filter low-confidence predictions
    entities = [e for e in entities if e["score"] >= threshold]

    if not entities:
        print("  No entities found above confidence threshold.")
        return []

    for entity in entities:
        print(
            f"  [{entity['entity_group']:5s}]  "
            f"'{entity['word']}'  "
            f"(confidence: {entity['score']:.3f})"
        )

    return entities


# ─────────────────────────────────────────────
# 14. RANDOM INFERENCE FROM DATASET
# ─────────────────────────────────────────────

import random

print("\n" + "=" * 62)
print("  NER INFERENCE RESULTS")
print("=" * 62)

num_samples = 7

random_indices = random.sample(range(len(dataset["test"])), num_samples)

for idx in random_indices:
    tokens = dataset["test"][idx]["tokens"]
    sentence = " ".join(tokens)
    run_ner(sentence, threshold=0.85)


  NER INFERENCE RESULTS

Input : The most notable use of the name in this context was by Bob Kane in naming the home of Batman , Gotham City .
--------------------------------------------------------------
  [PER  ]  'Bob Kane'  (confidence: 0.999)
  [PER  ]  'Batman'  (confidence: 0.973)

Input : He was later the player / manager of the Edmonton Eskimos in the Western Canada League in 1920 .
--------------------------------------------------------------
  [ORG  ]  'Edmonton Eskimos'  (confidence: 0.999)
  [ORG  ]  'Western Canada League'  (confidence: 0.998)

Input : He died in Richmond , Virginia , February 4 , 1908 .
--------------------------------------------------------------
  [LOC  ]  'Richmond , Virginia'  (confidence: 0.998)

Input : Corbu , Harghita
--------------------------------------------------------------
  [LOC  ]  'Corbu , Harghita'  (confidence: 0.999)

Input : Taypi Chaka Quta
--------------------------------------------------------------
  [PER  ]  'Taypi Chaka Q

In [ ]:
|